# Lockout Analysis

**Lockout** in Killer Queen: one team controls all 3 warrior gates (wings maidens),
has 2+ warriors, and the opponent has zero warriors — leaving them unable to upgrade.

Data: only games where at least one logged-in Hivemind user participated (`logged_in_games/`).

Questions:
- How often does lockout happen?
- How decisive is it? (win rate)
- Does team skill, queen skill, or skill differential matter?
- Does it vary by map?
- Does it vary by tournament vs casual play?

In [ ]:
import collections
import os
import sys
import pickle
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Find repo root by walking up until we find event_codec.py.
# Idempotent — safe to run this cell multiple times.
_repo_root = os.path.abspath('.')
while _repo_root != '/':
    if os.path.exists(os.path.join(_repo_root, 'event_codec.py')):
        break
    _repo_root = os.path.dirname(_repo_root)
assert os.path.exists(os.path.join(_repo_root, 'event_codec.py')), \
    f'Could not find repo root from {os.path.abspath(".")} — restart kernel with correct working directory'
os.chdir(_repo_root)
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)
print(f'Working directory: {os.getcwd()}')

from event_codec import read_packed_games, walk_game_states
from constants import ContestableState, Team, Map

# Wings maiden indices are {2, 3, 4} on all maps.
WINGS_INDICES = {2, 3, 4}

PACKED_PATH = 'logged_in_games/encoded/all_games.bin'

PLOTS_DIR = 'lockout_analysis/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)


def check_lockout(game_state, wings_indices=WINGS_INDICES):
    """Return the Team that has lockout, or None."""
    for team in [Team.BLUE, Team.GOLD]:
        own_warriors = sum(w.has_wings for w in game_state.teams[team.value].workers)
        opp = 1 - team.value
        opp_warriors = sum(w.has_wings for w in game_state.teams[opp].workers)
        team_cs = ContestableState.BLUE if team == Team.BLUE else ContestableState.GOLD
        all_wings_controlled = all(
            game_state.maiden_states[i] == team_cs for i in wings_indices
        )
        if own_warriors >= 2 and opp_warriors == 0 and all_wings_controlled:
            return team
    return None

In [ ]:
# Scan games for lockout using binary codec
rows = []

for game_id, encoded in read_packed_games(PACKED_PATH):
    lockout_found = False
    lockout_info = None
    lockout_ended = False
    gs = None
    last_ts = None

    for rel_ts, gs in walk_game_states(encoded):
        last_ts = rel_ts
        if not lockout_found:
            lockout_team = check_lockout(gs)
            if lockout_team is not None:
                lockout_found = True
                own_warriors = sum(
                    w.has_wings for w in gs.teams[lockout_team.value].workers
                )
                victim_idx = 1 - lockout_team.value
                lockout_info = {
                    'lockout_team': lockout_team.name.lower(),
                    'lockout_timestamp': rel_ts,
                    'lockout_warriors': own_warriors,
                    'victim_queen_eggs': gs.teams[victim_idx].eggs,
                }
        elif not lockout_ended:
            # Lockout duration ends when the victim team forms a warrior,
            # not when check_lockout() stops returning True. The victim
            # queen can tag a gate or kill an opposing warrior, but as long
            # as the victim team has zero warriors they're still locked out.
            victim_idx = 1 - lockout_team.value
            victim_warriors = sum(w.has_wings for w in gs.teams[victim_idx].workers)
            if victim_warriors > 0:
                lockout_ended = True
                lockout_info['lockout_end_timestamp'] = rel_ts

    # If lockout never ended, use last event timestamp
    if lockout_found and not lockout_ended:
        lockout_info['lockout_end_timestamp'] = last_ts

    # After walk, gs has winning_team/victory_condition set
    if gs is None or gs.winning_team is None:
        continue

    row = {
        'game_id': game_id,
        'map_name': gs.map_info.map_id.value,
        'blue_wins': int(gs.winning_team == Team.BLUE),
        'win_condition': gs.victory_condition.value,
        'game_duration': last_ts,
        'lockout_team': None,
        'lockout_timestamp': None,
        'lockout_warriors': None,
        'lockout_end_timestamp': None,
        'victim_queen_eggs': None,
    }
    if lockout_found:
        row.update(lockout_info)
    rows.append(row)

print(f'Collected {len(rows)} games')

In [ ]:
# Build DataFrame & basic stats
df = pd.DataFrame(rows)
df['has_lockout'] = df['lockout_team'].notna()

# For lockout games, did the lockout team win?
lockout_df = df[df['has_lockout']].copy()
lockout_df['lockout_team_wins'] = (
    ((lockout_df['lockout_team'] == 'blue') & (lockout_df['blue_wins'] == 1)) |
    ((lockout_df['lockout_team'] == 'gold') & (lockout_df['blue_wins'] == 0))
)

total_games = len(df)
lockout_games = df['has_lockout'].sum()
lockout_pct = lockout_games / total_games * 100
lockout_win_rate = lockout_df['lockout_team_wins'].mean() * 100

print(f'Total games: {total_games}')
print(f'Games with lockout: {lockout_games} ({lockout_pct:.1f}%)')
print(f'Lockout team win rate: {lockout_win_rate:.1f}%')
print()
print('Win condition breakdown (lockout games):')
print(lockout_df['win_condition'].value_counts())
print()
print('Lockout warrior count distribution:')
print(lockout_df['lockout_warriors'].value_counts().sort_index())

Total games: 182576
Games with lockout: 105508 (57.8%)
Lockout team win rate: 74.0%

Win condition breakdown (lockout games):
win_condition
military    77294
economic    18700
snail        9514
Name: count, dtype: int64

Lockout warrior count distribution:
lockout_warriors
2.0    67308
3.0    36271
4.0     1929
Name: count, dtype: int64


In [ ]:
# Win condition comparison: all games vs lockout games (with 95% CIs)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

wc_order = ['military', 'economic', 'snail']

def proportion_ci(counts, total):
    """95% CI half-width for proportions (percentage scale)."""
    p = counts / total
    return 1.96 * np.sqrt(p * (1 - p) / total) * 100

# All games
all_wc = df['win_condition'].value_counts().reindex(wc_order)
all_wc_pct = all_wc / len(df) * 100
all_ci = proportion_ci(all_wc.values, len(df))
all_wc_pct.plot.bar(ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title(f'All Games (n={len(df):,})')
axes[0].set_ylabel('% of games')
axes[0].tick_params(axis='x', rotation=0)
for i, (pct, ci) in enumerate(zip(all_wc_pct, all_ci)):
    axes[0].annotate(f'{pct:.1f} \u00b1 {ci:.1f}%', (i, pct), textcoords='offset points',
                     xytext=(0, 5), ha='center', fontsize=10)

# Lockout games
lo_wc = lockout_df['win_condition'].value_counts().reindex(wc_order)
lo_wc_pct = lo_wc / len(lockout_df) * 100
lo_ci = proportion_ci(lo_wc.values, len(lockout_df))
lo_wc_pct.plot.bar(ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title(f'Lockout Games (n={len(lockout_df):,})')
axes[1].set_ylabel('% of games')
axes[1].tick_params(axis='x', rotation=0)
for i, (pct, ci) in enumerate(zip(lo_wc_pct, lo_ci)):
    axes[1].annotate(f'{pct:.1f} \u00b1 {ci:.1f}%', (i, pct), textcoords='offset points',
                     xytext=(0, 5), ha='center', fontsize=10)

# Match y-axis scale
max_y = max(axes[0].get_ylim()[1], axes[1].get_ylim()[1])
axes[0].set_ylim(0, max_y * 1.1)
axes[1].set_ylim(0, max_y * 1.1)

plt.suptitle('Win Condition Breakdown: All Games vs Lockout Games', y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/win_condition_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Lockout by map (with 95% CIs)
map_stats = df.groupby('map_name').agg(
    total=('has_lockout', 'count'),
    lockout_count=('has_lockout', 'sum'),
).assign(lockout_pct=lambda x: x['lockout_count'] / x['total'] * 100)

map_win = lockout_df.groupby('map_name')['lockout_team_wins'].agg(['mean', 'count'])
map_win.columns = ['lockout_win_rate', 'n_lockout']
map_win['lockout_win_rate'] *= 100

map_combined = map_stats.join(map_win)
print(map_combined)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Lockout frequency by map with CIs
freq_pct = map_combined['lockout_pct']
freq_ci = 1.96 * np.sqrt(freq_pct/100 * (1 - freq_pct/100) / map_combined['total']) * 100
freq_pct.plot.bar(ax=axes[0], color='steelblue', yerr=freq_ci, capsize=4)
axes[0].set_title('Lockout Frequency by Map')
axes[0].set_ylabel('% of games with lockout')
axes[0].tick_params(axis='x', rotation=0)

# Lockout win rate by map with CIs
wr = map_combined['lockout_win_rate']
wr_ci = 1.96 * np.sqrt(wr/100 * (1 - wr/100) / map_combined['n_lockout']) * 100
wr.plot.bar(ax=axes[1], color='coral', yerr=wr_ci, capsize=4)
axes[1].set_title('Lockout Win Rate by Map')
axes[1].set_ylabel('Win rate (%)')
axes[1].axhline(50, color='gray', linestyle='--', alpha=0.5)
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/lockout_by_map.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Join with ratings
ratings_by_game = pickle.load(open('ratings.pkl', 'rb'))

# Ratings indices: gold_queen=0, blue_queen=1,
# gold_drones=2,4,6,8, blue_drones=3,5,7,9

def get_team_ratings(game_id, team_str):
    """Return (queen_mu, avg_drone_mu) for the given team."""
    r = ratings_by_game.get(game_id)
    if r is None:
        return None, None
    if team_str == 'blue':
        queen_mu = float(r[1])
        drone_mus = [float(r[i]) for i in [3, 5, 7, 9]]
    else:
        queen_mu = float(r[0])
        drone_mus = [float(r[i]) for i in [2, 4, 6, 8]]
    return queen_mu, np.mean(drone_mus)


lockout_ratings = []
for _, row in lockout_df.iterrows():
    lo_team = row['lockout_team']
    victim_team = 'gold' if lo_team == 'blue' else 'blue'
    lo_queen, lo_drone_avg = get_team_ratings(row['game_id'], lo_team)
    vi_queen, vi_drone_avg = get_team_ratings(row['game_id'], victim_team)
    lockout_ratings.append({
        'game_id': row['game_id'],
        'lo_queen_mu': lo_queen,
        'lo_drone_avg_mu': lo_drone_avg,
        'vi_queen_mu': vi_queen,
        'vi_drone_avg_mu': vi_drone_avg,
    })

ratings_df = pd.DataFrame(lockout_ratings)
lo = lockout_df.merge(ratings_df, on='game_id')
lo = lo.dropna(subset=['lo_queen_mu'])
lo['skill_diff'] = (lo['lo_queen_mu'] + lo['lo_drone_avg_mu']) - (lo['vi_queen_mu'] + lo['vi_drone_avg_mu'])
lo['queen_diff'] = lo['lo_queen_mu'] - lo['vi_queen_mu']
print(f'Lockout games with ratings: {len(lo)}')
lo[['lo_queen_mu', 'lo_drone_avg_mu', 'vi_queen_mu', 'vi_drone_avg_mu', 'skill_diff']].describe()

FileNotFoundError: [Errno 2] No such file or directory: 'ratings.pkl'

In [ ]:
# Lockout win rate by skill differential
lo['skill_diff_bin'] = pd.qcut(lo['skill_diff'], q=5, duplicates='drop')

skill_win = lo.groupby('skill_diff_bin', observed=True)['lockout_team_wins'].agg(['mean', 'count'])
skill_win.columns = ['win_rate', 'n']

fig, ax = plt.subplots(figsize=(8, 4))
midpoints = [interval.mid for interval in skill_win.index]
ax.plot(midpoints, skill_win['win_rate'], marker='o', color='steelblue')
ax.set_title('Lockout Win Rate by Skill Differential (lockout team - victim)')
ax.set_ylabel('Win rate')
ax.set_xlabel('Skill diff (lockout team advantage)')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
for x, (_, row) in zip(midpoints, skill_win.iterrows()):
    ax.annotate(f'n={row["n"]:.0f}', (x, row['win_rate']), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/winrate_by_skill_diff.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey question: does lockout override a skill deficit?')
behind = lo[lo['skill_diff'] < 0]
print(f'Lockout team is lower-skilled: {len(behind)} games, win rate: {behind["lockout_team_wins"].mean():.1%}')

In [ ]:
# Lockout win rate by queen skill
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Victim queen skill
lo['vi_queen_bin'] = pd.qcut(lo['vi_queen_mu'], q=5, duplicates='drop')
vq = lo.groupby('vi_queen_bin', observed=True)['lockout_team_wins'].agg(['mean', 'count'])
midpoints_vq = [interval.mid for interval in vq.index]
axes[0].plot(midpoints_vq, vq['mean'], marker='o', color='coral')
axes[0].set_title('Lockout Win Rate by Victim Queen Skill')
axes[0].set_ylabel('Win rate')
axes[0].set_xlabel('Victim queen mu')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
for x, (_, row) in zip(midpoints_vq, vq.iterrows()):
    axes[0].annotate(f'n={row["count"]:.0f}', (x, row['mean']), textcoords='offset points',
                     xytext=(0, 10), ha='center', fontsize=9)

# Lockout team queen skill
lo['lo_queen_bin'] = pd.qcut(lo['lo_queen_mu'], q=5, duplicates='drop')
lq = lo.groupby('lo_queen_bin', observed=True)['lockout_team_wins'].agg(['mean', 'count'])
midpoints_lq = [interval.mid for interval in lq.index]
axes[1].plot(midpoints_lq, lq['mean'], marker='o', color='steelblue')
axes[1].set_title('Lockout Win Rate by Lockout Team Queen Skill')
axes[1].set_ylabel('Win rate')
axes[1].set_xlabel('Lockout team queen mu')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
for x, (_, row) in zip(midpoints_lq, lq.iterrows()):
    axes[1].annotate(f'n={row["count"]:.0f}', (x, row['mean']), textcoords='offset points',
                     xytext=(0, 10), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/winrate_by_queen_skill.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Lockout punitiveness by skill level for evenly-matched games
# "Even" = skill_diff within +/- 1 standard deviation of the median (~0)
skill_diff_std = lo['skill_diff'].std()
even = lo[lo['skill_diff'].abs() < skill_diff_std].copy()
print(f'Even-skill games (|skill_diff| < {skill_diff_std:.1f}): {len(even)}')

# Average team skill = mean of lockout + victim team ratings
even['avg_skill'] = (lo['lo_queen_mu'] + lo['lo_drone_avg_mu'] +
                     lo['vi_queen_mu'] + lo['vi_drone_avg_mu']) / 2
even['skill_bin'] = pd.qcut(even['avg_skill'], q=6, duplicates='drop')

skill_level = even.groupby('skill_bin', observed=True)['lockout_team_wins'].agg(['mean', 'count'])
midpoints = [interval.mid for interval in skill_level.index]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(midpoints, skill_level['mean'], marker='o', color='steelblue')
ax.set_title('Lockout Win Rate by Skill Level (even-matched games only)')
ax.set_ylabel('Lockout team win rate')
ax.set_xlabel('Average team skill (queen + drone avg)')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
for x, (_, row) in zip(midpoints, skill_level.iterrows()):
    ax.annotate(f'n={row["count"]:.0f}', (x, row['mean']), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/punitiveness_by_skill.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nLowest skill bin win rate: {skill_level["mean"].iloc[0]:.1%}')
print(f'Highest skill bin win rate: {skill_level["mean"].iloc[-1]:.1%}')

In [ ]:
# Temporal analysis: when does lockout first occur and how decisive is early vs late?
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution of lockout timestamps (clamp to 0-300s)
lockout_ts = lockout_df['lockout_timestamp'].dropna()
lockout_ts_clamped = lockout_ts[lockout_ts <= 300]
axes[0].hist(lockout_ts_clamped, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('When Does Lockout First Occur?')
axes[0].set_xlabel('Game time (seconds)')
axes[0].set_ylabel('Count')
axes[0].set_xlim(0, 300)
axes[0].axvline(lockout_ts.median(), color='red', linestyle='--', label=f'Median={lockout_ts.median():.0f}s')
axes[0].legend()

# Win rate by lockout time bin (line graph)
lo_time = lockout_df.copy()
lo_time['time_bin'] = pd.qcut(lo_time['lockout_timestamp'], q=5, duplicates='drop')
time_win = lo_time.groupby('time_bin', observed=True)['lockout_team_wins'].agg(['mean', 'count'])
midpoints_t = [interval.mid for interval in time_win.index]
axes[1].plot(midpoints_t, time_win['mean'], marker='o', color='coral')
axes[1].set_title('Lockout Win Rate by Time of First Lockout')
axes[1].set_ylabel('Win rate')
axes[1].set_xlabel('Time of first lockout (seconds)')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
for x, (_, row) in zip(midpoints_t, time_win.iterrows()):
    axes[1].annotate(f'n={row["count"]:.0f}', (x, row['mean']), textcoords='offset points',
                     xytext=(0, 10), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/lockout_timing.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Median time of first lockout: {lockout_ts.median():.0f}s')
print(f'First lockout after 300s: {(lockout_ts > 300).sum()}')
early = lockout_df[lockout_df['lockout_timestamp'] < lockout_ts.median()]
late = lockout_df[lockout_df['lockout_timestamp'] >= lockout_ts.median()]
print(f'Early lockout win rate: {early["lockout_team_wins"].mean():.1%} (n={len(early)})')
print(f'Late lockout win rate: {late["lockout_team_wins"].mean():.1%} (n={len(late)})')

In [ ]:
# Conditional lockout probability: given the game has reached time T
# without lockout, what's the chance lockout happens in the next 5 seconds?
bin_width = 5
max_time = 180  # focus on first 3 minutes
bins = np.arange(0, max_time, bin_width)

hazard_rates = []
for t in bins:
    # Games "at risk": still going at time t and no lockout yet
    at_risk = ((df['game_duration'] >= t) &
               (df['lockout_timestamp'].isna() | (df['lockout_timestamp'] >= t))).sum()
    # Lockouts in [t, t + bin_width)
    events = ((df['lockout_timestamp'] >= t) &
              (df['lockout_timestamp'] < t + bin_width)).sum()
    rate = events / at_risk if at_risk > 0 else 0
    ci = 1.96 * np.sqrt(rate * (1 - rate) / at_risk) if at_risk > 0 else 0
    hazard_rates.append({'time': t + bin_width / 2, 'rate': rate, 'ci': ci,
                         'at_risk': at_risk, 'events': events})

hazard_df = pd.DataFrame(hazard_rates)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(hazard_df['time'], hazard_df['rate'] * 100,
       yerr=hazard_df['ci'] * 100, capsize=2,
       width=bin_width * 0.8, color='steelblue', edgecolor='white')
ax.set_title('Conditional Lockout Probability per 5-Second Window')
ax.set_xlabel('Game time (seconds)')
ax.set_ylabel('P(lockout in next 5s | no lockout yet) %')
ax.set_xlim(0, max_time)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/conditional_lockout_hazard.png', dpi=150, bbox_inches='tight')
plt.show()

peak_idx = hazard_df['rate'].idxmax()
print(f'Peak hazard: {hazard_df.loc[peak_idx, "rate"]*100:.1f}% at '
      f't={hazard_df.loc[peak_idx, "time"]:.0f}s')
print(f'Games still at risk at 60s: {hazard_df.loc[hazard_df["time"]==62.5, "at_risk"].values[0]:,}')
print(f'Games still at risk at 120s: {hazard_df.loc[hazard_df["time"]==122.5, "at_risk"].values[0]:,}')

In [ ]:
# Lockout duration analysis
# Duration = time from first lockout until victim team forms a warrior.
# If lockout persists until game end, duration extends to the final event.
lockout_df['lockout_duration'] = lockout_df['lockout_end_timestamp'] - lockout_df['lockout_timestamp']
duration = lockout_df['lockout_duration'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: histogram of lockout duration (clamp to 0-300s for readability)
duration_clamped = duration[duration <= 300]
axes[0].hist(duration_clamped, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Duration of First Lockout')
axes[0].set_xlabel('Lockout duration (seconds)')
axes[0].set_ylabel('Count')
axes[0].axvline(duration.median(), color='red', linestyle='--', label=f'Median={duration.median():.0f}s')
axes[0].legend()

# Right: win rate by duration bin
lo_dur = lockout_df.dropna(subset=['lockout_duration']).copy()
lo_dur['duration_bin'] = pd.qcut(lo_dur['lockout_duration'], q=5, duplicates='drop')
dur_win = lo_dur.groupby('duration_bin', observed=True)['lockout_team_wins'].agg(['mean', 'count'])
midpoints_d = [interval.mid for interval in dur_win.index]
axes[1].plot(midpoints_d, dur_win['mean'], marker='o', color='coral')
axes[1].set_title('Lockout Win Rate by Duration of First Lockout')
axes[1].set_ylabel('Win rate')
axes[1].set_xlabel('Lockout duration (seconds)')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
for x, (_, row) in zip(midpoints_d, dur_win.iterrows()):
    axes[1].annotate(f'n={row["count"]:.0f}', (x, row['mean']), textcoords='offset points',
                     xytext=(0, 10), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/lockout_duration.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Median lockout duration: {duration.median():.0f}s')
print(f'Mean lockout duration: {duration.mean():.0f}s')
short = (duration < 5).sum()
long = (duration > 60).sum()
print(f'Lockouts lasting < 5s: {short} ({short/len(duration)*100:.1f}%)')
print(f'Lockouts lasting > 60s: {long} ({long/len(duration)*100:.1f}%)')

In [ ]:
# Victim win probability by queen eggs remaining at the start of lockout
# eggs=2: queen hasn't died, eggs=1: died once, eggs=0: died twice
eggs_df = lockout_df.dropna(subset=['victim_queen_eggs']).copy()
eggs_df['victim_wins'] = ~eggs_df['lockout_team_wins']

egg_stats = eggs_df.groupby('victim_queen_eggs').agg(
    victim_win_rate=('victim_wins', 'mean'),
    n=('victim_wins', 'count'),
)
egg_stats['ci'] = 1.96 * np.sqrt(
    egg_stats['victim_win_rate'] * (1 - egg_stats['victim_win_rate']) / egg_stats['n']
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(egg_stats.index, egg_stats['victim_win_rate'] * 100,
       yerr=egg_stats['ci'] * 100, capsize=6, color='coral', edgecolor='white')
ax.set_title('Victim Win Probability by Queen Lives at Lockout')
ax.set_xlabel('Victim queen eggs remaining')
ax.set_ylabel('Victim win rate (%)')
ax.set_xticks(egg_stats.index)
ax.axhline(50, color='gray', linestyle='--', alpha=0.5)
for idx, row in egg_stats.iterrows():
    ax.annotate(f'{row["victim_win_rate"]*100:.1f}%\n(n={row["n"]:,.0f})',
                (idx, (row["victim_win_rate"] + row["ci"]) * 100),
                textcoords='offset points', xytext=(0, 5), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/victim_queen_eggs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tournament analysis
game_meta_df = pd.read_csv('unfiltered_partitioned/game.csv')
game_meta_df = game_meta_df[['id', 'tournament_match_id']].rename(columns={'id': 'game_id'})
game_meta_df['is_tournament'] = game_meta_df['tournament_match_id'].notna()

df2 = df.merge(game_meta_df, on='game_id', how='left')
df2['is_tournament'] = df2['is_tournament'].fillna(False)

# Load tournament clustering for stage info
with open('late_tournament_games/tournament_clustering.json') as f:
    tourney_data = json.load(f)
match_to_tournament = tourney_data['match_to_tournament']
tournaments = tourney_data['tournaments']

# Compare lockout frequency: tournament vs casual
for label, subset in [('Tournament', df2[df2['is_tournament']]),
                       ('Casual', df2[~df2['is_tournament']])]:
    n = len(subset)
    n_lockout = subset['has_lockout'].sum()
    pct = n_lockout / n * 100 if n > 0 else 0
    lockout_sub = subset[subset['has_lockout']]
    # Recompute lockout_team_wins for this subset
    if len(lockout_sub) > 0:
        wins = (
            ((lockout_sub['lockout_team'] == 'blue') & (lockout_sub['blue_wins'] == 1)) |
            ((lockout_sub['lockout_team'] == 'gold') & (lockout_sub['blue_wins'] == 0))
        ).mean() * 100
    else:
        wins = float('nan')
    print(f'{label}: {n} games, {n_lockout} lockouts ({pct:.1f}%), lockout win rate: {wins:.1f}%')

Tournament: 29583 games, 19546 lockouts (66.1%), lockout win rate: 78.1%
Casual: 153376 games, 86014 lockouts (56.1%), lockout win rate: 73.1%
